In [0]:
# 1. Define your Azure Data Lake details
storage_account_name = "adlsstreamanalytics01"
container_name = "bronze"

# 2. Define your Service Principal credentials (from your Notepad)
client_id = ""
tenant_id = ""
client_secret = ""

# 3. Configure the Spark session to securely authenticate using OAuth 2.0
spark.conf.set(f"fs.azure.account.auth.type.{storage_account_name}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{storage_account_name}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{storage_account_name}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{storage_account_name}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{storage_account_name}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")

# 4. Define the exact path to your raw JSON file
file_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/raw_streaming_events.json"

# 5. Read the file into a DataFrame and display it!
df_bronze = spark.read.json(file_path)
df_bronze.display()

artist_id,country,device_type,event_id,event_timestamp,song_id,stream_duration_sec,track_name,user_id
A01,DE,iOS,6f5fb3f4-7215-483e-84f9-111c8c40a78c,2026-09-29T02:13:02.482530+00:00,S104,144,Echoes in Rain,USR_1048
A02,US,Web,07fa0d17-3433-44e0-a8e1-bcfb84c6d6a0,2026-09-29T02:13:02.482550+00:00,S102,69,Golden Horizon,USR_1019
A02,US,Android,4e357c0b-9ca4-4a8a-8162-23db292d1a24,2026-09-29T02:13:02.482557+00:00,S102,188,Golden Horizon,USR_1012
A03,US,iOS,904aaf17-b11a-4e00-a00e-3e9a2674a3bd,2026-09-29T02:13:02.482563+00:00,S103,162,Urban Beats,USR_1004
A03,CA,Desktop,bf208582-a6e5-4157-b01a-efd6811e9e8c,2026-09-29T02:13:02.482570+00:00,S103,240,Urban Beats,USR_1029
A02,US,Desktop,d5353111-a0ea-47ed-b6dd-df9c2dfbcdde,2026-09-29T02:13:02.482576+00:00,S102,11,Golden Horizon,USR_1050
A02,IN,Android,4568c51e-2194-42d8-9c71-5c0fb25e1122,2026-09-29T02:13:02.482582+00:00,S102,132,Golden Horizon,USR_1044
A02,CA,Android,f9fef406-09ae-4aef-a022-03720ab14a2a,2026-09-29T02:13:02.482586+00:00,S102,240,Golden Horizon,USR_1031
A01,CA,Android,367dd491-295a-4e6b-9bf9-890440a3890b,2026-09-29T02:13:02.482592+00:00,S104,178,Echoes in Rain,USR_1041
A02,CA,Desktop,adcb1ed9-73e8-4e33-9e31-279ddb01fedc,2026-09-29T02:13:02.482597+00:00,S102,50,Golden Horizon,USR_1024


In [0]:
from pyspark.sql.functions import col, to_timestamp

# 1. Cleanse & Transform: Remove duplicates based on event_id and cast the timestamp
df_silver = df_bronze.dropDuplicates(["event_id"]) \
                     .withColumn("event_timestamp", to_timestamp(col("event_timestamp")))

# 2. Define the exact path to your 'silver' container
silver_path = f"abfss://silver@{storage_account_name}.dfs.core.windows.net/streaming_events_silver"

# 3. Write the transformed data into the Silver container as a Delta table
df_silver.write.format("delta").mode("overwrite").save(silver_path)

print(f"Success! Data transformed and written to: {silver_path}")

# 4. Read it back from Silver to verify
df_verify_silver = spark.read.format("delta").load(silver_path)
df_verify_silver.display()

Success! Data transformed and written to: abfss://silver@adlsstreamanalytics01.dfs.core.windows.net/streaming_events_silver


artist_id,country,device_type,event_id,event_timestamp,song_id,stream_duration_sec,track_name,user_id
A01,GB,Android,f99ac818-7ca8-4fda-b4ea-21484f0e02f9,2026-09-29T02:13:02.482723Z,S104,41,Echoes in Rain,USR_1046
A02,US,Web,07fa0d17-3433-44e0-a8e1-bcfb84c6d6a0,2026-09-29T02:13:02.48255Z,S102,69,Golden Horizon,USR_1019
A03,DE,iOS,fcdc30c5-8ae0-4de4-85b9-66955ea518ee,2026-09-29T02:13:02.482648Z,S103,53,Urban Beats,USR_1010
A03,GB,Web,eb4f1b44-f7da-4283-936e-5ea7e4fbc467,2026-09-29T02:13:02.482698Z,S103,224,Urban Beats,USR_1040
A01,IN,iOS,6e2d3710-def9-4854-b2c2-b4f56b2cfc16,2026-09-29T02:13:02.482737Z,S104,217,Echoes in Rain,USR_1017
A03,IN,Desktop,520840c4-d6ca-43ac-93b2-efdcd9b82cbb,2026-09-29T02:13:02.482768Z,S103,101,Urban Beats,USR_1027
A03,US,iOS,525d639d-d34e-4b3c-a950-ff47e6947df9,2026-09-29T02:13:02.482601Z,S103,37,Urban Beats,USR_1032
A02,IN,Desktop,4560583e-1ca4-424f-b54d-22c8f7707eaa,2026-09-29T02:13:02.482653Z,S102,48,Golden Horizon,USR_1032
A01,IN,iOS,cab1afa6-bee0-48ea-bc63-8fb930ce3050,2026-09-29T02:13:02.482793Z,S101,62,Midnight Drive,USR_1039
A03,US,Web,76633caf-c83b-4b03-98da-0814bc676ef3,2026-09-29T02:13:02.482759Z,S103,168,Urban Beats,USR_1037


In [0]:
from pyspark.sql.functions import count, sum, col

# 1. Read the cleansed data from the Silver layer
df_silver = spark.read.format("delta").load(silver_path)

# 2. Aggregate data for the Gold layer: Total streams and duration per song
df_gold_songs = df_silver.groupBy("song_id", "track_name", "artist_id") \
                         .agg(
                             count("event_id").alias("total_streams"),
                             sum("stream_duration_sec").alias("total_duration_sec")
                         ) \
                         .orderBy(col("total_streams").desc())

# 3. Define the path to your 'gold' container
gold_path = f"abfss://gold@{storage_account_name}.dfs.core.windows.net/gold_song_metrics"

# 4. Write the aggregated data to Gold as a Delta table
df_gold_songs.write.format("delta").mode("overwrite").save(gold_path)

print(f"Success! Business aggregates written to: {gold_path}")

# 5. Display the final business-ready Gold data
df_gold_songs.display()

Success! Business aggregates written to: abfss://gold@adlsstreamanalytics01.dfs.core.windows.net/gold_song_metrics


song_id,track_name,artist_id,total_streams,total_duration_sec
S103,Urban Beats,A03,17,2275
S102,Golden Horizon,A02,13,1607
S101,Midnight Drive,A01,10,1548
S104,Echoes in Rain,A01,10,1188


In [0]:
# 1. Save the detailed Silver data (this contains country, device_type, etc.)
df_silver.write.mode("overwrite").option("mergeSchema", "true").saveAsTable("default.silver_music_data")

# 2. Save the aggregated Gold data (this contains total_streams, track_name, etc.)
df_gold_songs.write.mode("overwrite").saveAsTable("default.gold_music_data")